# In-memory inputs: the two adapters of ADR-0017

Every pipeline input is loadable through **exactly two adapters** that converge on one
in-memory contract:

- the **file adapter** — a path in `InputParams`, parsed by a low-level reader;
- the **injection adapter** — the in-memory object handed straight to the high-level class.

The injected object is defined as *exactly what the reader would have returned* — no new
flexibility in accepted forms. This notebook drives both adapters over the same data and
asserts they produce **identical** spectra.

This is the path a pipeline integration takes: it already holds maps, a mask and a noise
covariance in memory, and should not have to round-trip them through disk to use QUBE.

Injection kwargs (the closed vocabulary): `mask`, `noise_cov1`/`noise_cov2`,
`maps1`/`maps2`, `cls_data`, `fiducial_cls`, `beam`.


In [ ]:
import os
import tempfile

import healpy as hp
import numpy as np
import yaml

from cosmocore import InputParams
from cosmocore.in_out import read_covmat, read_maps, read_mask, readcl
from qube import Fisher, Spectra

QUBE_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))


def resolve_config(config_rel):
    """Anchor the YAML's relative paths to the qube root, so this runs from any cwd."""
    with open(os.path.join(QUBE_ROOT, config_rel)) as f:
        config = yaml.safe_load(f)
    for key, value in config.items():
        if isinstance(value, str):
            clean = value
            while clean.startswith("../"):
                clean = clean[3:]
            if clean.startswith(("tests/", "scripts/")):
                config[key] = os.path.join(QUBE_ROOT, clean)
    tmp = tempfile.NamedTemporaryFile(mode="w", suffix=".yaml", delete=False)
    yaml.dump(config, tmp)
    tmp.close()
    return tmp.name


param_file = resolve_config("tests/data/nside4/T/config.yaml")
params = InputParams.read_parameter_file(param_file)

# Disk-free: clear every output-artifact path, so nothing is written (ADR-0015).
for _k in (
    "output_geometry_file",
    "outnoisecovmat1",
    "outnoisecovmat2",
    "outinvcovmatfile1",
    "outinvcovmatfile2",
    "outfilefisher",
    "outcovmatfile",
    "outerrfile",
):
    setattr(params, _k, "")

params.nsims = 3
lmax_signal = 4 * params.nside
print(f"nside={params.nside}  lmax={params.lmax}  lmax_signal={lmax_signal}")

## 1. File adapter — the reference run

The ordinary config-driven path: every input is a path in the YAML. This is unchanged by
ADR-0017 and remains the workflow for HPC batch jobs.

In [ ]:
fisher_file = Fisher(params)
fisher_file.run()

spectra_file = Spectra(params, fisher=fisher_file)
spectra_file.run()
ps_file = spectra_file.get_power_spectra(mode="deconvolved")

print(f"file adapter  -> power spectra {ps_file.shape}")

## 2. Low-level readers → in-memory arrays

The low-level layer (`cosmocore.in_out`, `healpy`) is importable and callable *without* the
framework: readers are pure parsers, path → array. Here we use them directly to produce
exactly the objects the injection adapter expects.

The contract per seam is "what the reader returns":

| kwarg | reader | note |
|---|---|---|
| `mask` | `read_mask` | `(npix, nfields)`, ordered per `params.ordering` |
| `cls_data`, `fiducial_cls` | `readcl` | `{label: C_ell}` dict, **physical** C_ell |
| `beam` | `hp.read_cl` | 2-D, ≥3 rows (T/E/B); pixwin already folded in |
| `noise_cov1` | `read_covmat` | **reduced** `(n_active, n_active)`, **pre-calibration** |
| `maps1` | `read_maps` | reduced `(n_active, n_sims)`, **already calibrated** |

Note the two asymmetric ones — they are not arbitrary, they mirror what each reader does:
`read_covmat` does *not* apply `calibration` (the framework applies `calibration**2`
afterwards), whereas `read_maps` *does* apply it on read.

In [ ]:
nfields = params.nfields
npix = hp.nside2npix(params.nside)

# read_mask only reads shape[0] (the field count) off this argument.
shape_proxy = np.empty((nfields, 0), dtype=np.float64)
mask = read_mask(params.maskfile, shape_proxy, nest=params.ordering == "NESTED")

cls = readcl(params.inputclfile, params, lmax=lmax_signal)
fiducial_cls = readcl(
    getattr(params, "fiducialfile", None) or params.inputclfile, params, lmax=lmax_signal
)
beam = hp.read_cl(params.beam_file).astype(np.float64)

print(f"mask   {mask.shape}")
print(f"cls    {sorted(cls)}  (each length {len(next(iter(cls.values())))})")
print(f"beam   {beam.shape}")

### The covariance and the maps need the active-pixel index

`noise_cov1` and `maps1` are **reduced** to the active (unmasked) pixels, concatenated
across fields. So a caller must know the active-pixel ordering before it can build them —
which means the geometry has to be set up first.

That is a real seam in the contract: an integration holding its own covariance cannot hand
it over blind — it has to reduce it to the same active-pixel ordering. We take that ordering
from the geometry the framework already computed.

(The bundled test config used here happens to be **full sky**, so `n_active == npix` and the
reduction below is an identity. The code is unchanged for a cut sky — that is exactly the
step a masked-sky caller must perform, and it is why the geometry has to come first.)

In [ ]:
pixact = fisher_file.pixact  # active pixel indices, per field
concat_pixact = np.concatenate([pixact[i] + i * npix for i in range(len(pixact))])
n_active = concat_pixact.shape[0]
print(f"active pixels: {n_active} (of {npix * nfields})")

# Reduced noise covariance, PRE-calibration (the framework applies calibration**2).
noise_cov1 = read_covmat(
    params.covmatfile1,
    npix,
    nfields,
    concat_pixact,
    np.empty((n_active, n_active), dtype=np.float64),
)

# Reduced maps, ALREADY calibrated (read_maps applies calibration on read).
maps1 = np.empty((n_active, params.nsims), dtype=np.float64)
read_maps(
    maps=maps1,
    filename=params.inputmapfile1,
    pixact=pixact,
    field_labels=params.physical_labels,
    calibration=params.calibration,
)

print(f"noise_cov1  {noise_cov1.shape}")
print(f"maps1       {maps1.shape}")

## 3. Injection adapter — arrays in, nothing read from disk

Now point every input path at a location that does not exist, and hand the arrays in as
constructor kwargs. If anything were still being read from disk, this would raise.

In [ ]:
params_mem = InputParams.read_parameter_file(param_file)
for _k in (
    "output_geometry_file",
    "outnoisecovmat1",
    "outnoisecovmat2",
    "outinvcovmatfile1",
    "outinvcovmatfile2",
    "outfilefisher",
    "outcovmatfile",
    "outerrfile",
):
    setattr(params_mem, _k, "")
params_mem.nsims = params.nsims

# Every input path now points nowhere: the arrays are the only source of truth.
for _k in (
    "maskfile",
    "covmatfile1",
    "covmatfile2",
    "inputclfile",
    "fiducialfile",
    "beam_file",
    "inputmapfile1",
    "inputmapfile2",
):
    setattr(params_mem, _k, "/nonexistent/" + _k)

fisher_mem = Fisher(
    params_mem,
    mask=mask,
    noise_cov1=noise_cov1,
    cls_data=cls,
    fiducial_cls=fiducial_cls,
    beam=beam,
)
fisher_mem.run()

# maps are a Spectra/PICSLike seam (Fisher never reads them).
spectra_mem = Spectra(params_mem, fisher=fisher_mem, maps1=maps1)
spectra_mem.run()
ps_mem = spectra_mem.get_power_spectra(mode="deconvolved")

print(f"injection adapter -> power spectra {ps_mem.shape}  (no file was read)")

## 4. The two adapters agree

This is the ADR-0017 invariant: both adapters converge on one contract, so the estimated
spectra must be identical — not merely close.

In [ ]:
np.testing.assert_allclose(ps_mem, ps_file, rtol=1e-12, atol=0)
np.testing.assert_allclose(
    fisher_mem.get_fisher_matrix(), fisher_file.get_fisher_matrix(), rtol=1e-12, atol=0
)
print("file adapter == injection adapter  ✓")
print("  (identical Fisher matrix and identical power spectra)")

## What this buys you

A pipeline that already holds its sky maps, mask and noise covariance in memory can drive
QUBE without writing a single file — combine this with opt-in persistence (ADR-0015) and
the working directory stays untouched end to end.

The same kwargs exist on `PICSLike`. `Fisher` takes everything except the maps;
`maps1`/`maps2` are the `Spectra`/`PICSLike` seam, since `Fisher` never reads maps.

**Gotchas worth repeating:**

- `noise_cov1` is the **reduced** matrix (active pixels, concatenated across fields) and is
  **pre-calibration** — the framework multiplies by `calibration**2`.
- `maps1` is reduced too, and is **already calibrated** — `read_maps` applies it on read, so
  the framework does not re-apply it.
- `cls_data`/`fiducial_cls` are **physical** C_ell. `input_convention` (e.g. D_ell → C_ell)
  is applied only on the file path, never to injected arrays.
- An injected object always wins over the corresponding path in `InputParams`.